<h1 style="text-align:center;">Lab 7 — Transformer Fine-tuning · Обучение</h1>

Этот ноутбук рассчитан на запуск в Google Colab (Runtime → T4 GPU). На выходе сохраняются:
- `distilbert_imdb_ft/` — fine-tuned модель и токенизатор;
- `history.json` — лог обучения (loss / eval_accuracy по эпохам).

Оба артефакта нужно скачать и положить рядом с `Lab7_transformer_infer.ipynb`.

In [ ]:
!pip install -q transformers datasets evaluate scikit-learn 'accelerate>=1.1.0'

In [ ]:
import json
import numpy as np
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from sklearn.metrics import accuracy_score

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
print("Torch:", torch.__version__)

## Данные

IMDB (25k/25k отзывов, метки neg/pos). Для скорости берём подвыборку 4000/2000.

In [ ]:
raw = load_dataset("stanfordnlp/imdb")

TRAIN_N = 4000
TEST_N = 2000

train_ds = raw["train"].shuffle(seed=42).select(range(TRAIN_N))
test_ds = raw["test"].shuffle(seed=42).select(range(TEST_N))

print("Train:", len(train_ds), "Test:", len(test_ds))
print("Labels:", train_ds.features["label"])

for i in range(2):
    ex = train_ds[i]
    print(f"\n--- label={ex['label']} ---")
    print(ex["text"][:300], "...")

## Токенизация

Токенизатор от DistilBERT, `max_length=256`.

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

train_tok = train_ds.map(tokenize, batched=True).remove_columns(["text"])
test_tok = test_ds.map(tokenize, batched=True).remove_columns(["text"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print(train_tok)

## Модель: DistilBERT + классификационная голова

Берём `distilbert-base-uncased` и навешиваем классификатор на 2 класса (pre_classifier + classifier инициализируются с нуля).

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "neg", 1: "pos"},
    label2id={"neg": 0, "pos": 1},
)
model.to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Параметров в модели: {n_params/1e6:.1f}M")

## Fine-tuning

Гиперпараметры стандартные: lr=2e-5, weight_decay=0.01, 2 эпохи, batch_size=16.

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds)}

training_args = TrainingArguments(
    output_dir="./out",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    logging_steps=50,
    save_strategy="no",
    report_to="none",
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=test_tok,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

## Сохраняем артефакты

После этой ячейки нужно скачать `distilbert_imdb_ft/` и `history.json` из Colab (вкладка Files) и положить рядом с `Lab7_transformer_infer.ipynb`.

In [ ]:
SAVED_DIR = "distilbert_imdb_ft"

trainer.save_model(SAVED_DIR)
tokenizer.save_pretrained(SAVED_DIR)

with open("history.json", "w") as f:
    json.dump(trainer.state.log_history, f, indent=2)

print(f"Модель сохранена в '{SAVED_DIR}', история — в history.json")

In [ ]:
# (опционально) запаковать модель в zip, чтобы удобно скачать одним файлом
# !zip -r distilbert_imdb_ft.zip distilbert_imdb_ft
# from google.colab import files
# files.download('distilbert_imdb_ft.zip')
# files.download('history.json')